## Symbolic Calc Utils

In [1]:
from copy import deepcopy
from collections import defaultdict
from itertools import chain

import sympy
from sympy import Symbol, Expr, Function, symbols 

In [2]:
class SingleOp:
    def __init__(self, subscr: str, label: str, dagger: bool = False):
        self.subscr = subscr
        self.label = label
        self.dagger = dagger

    def _generate_latex_str(self):
        if self.dagger:
            return f"c_{{{self.subscr}}}^{{\\dagger}}({self.label})"
        else:
            return f"c_{{{self.subscr}}}({self.label})"

    def __repr__(self):
        return f"SingleOp(label={self.label}, subscr={self.subscr}, dagger={self.dagger})"

    def _repr_latex_(self):
        latex_str = self._generate_latex_str()
        return f"${latex_str}$"
    
    def __eq__(self, other):
        if not isinstance(other, SingleOp):
            return NotImplemented
        return (self.subscr, self.label, self.dagger) == (other.subscr, other.label, other.dagger)

    def __hash__(self):
        return hash((self.subscr, self.label, self.dagger))
    
    def dag(self):
        return SingleOp(self.subscr, self.label, not self.dagger)
    
class OpString:
    def __init__(self, ops: tuple[SingleOp]):
        self.ops = ops

    def _generate_latex_str(self):
        return " ".join([op._generate_latex_str() for op in self.ops])

    def __repr__(self):
        return f"OpString(ops={self.ops})"

    def _repr_latex_(self):
        latex_str = self._generate_latex_str()
        return f"${latex_str}$"
    
    def __len__(self):
        return len(self.ops)

    def __getitem__(self, idx):
        return self.ops[idx]

    def __eq__(self, other):
        if not isinstance(other, OpString):
            return NotImplemented
        return self.ops == other.ops

    def __hash__(self):
        return hash(self.ops)
    
    def dag(self):
        new_ops = [op.dag() for op in reversed(self.ops)]
        return OpString(tuple(new_ops))
    
    def is_pairable(self):
        n_creations = defaultdict(int)
        n_annihilations = defaultdict(int)
        
        for op in self.ops:
            key = (op.subscr, op.label)
            if op.dagger:
                n_creations[key] += 1
            else:
                n_annihilations[key] += 1
                
        n_creations = dict(n_creations)
        n_annihilations = dict(n_annihilations)

        if set(n_creations.keys()) != set(n_annihilations.keys()):
            return False
        
        if any(count > 1 for count in n_annihilations.values()):
            return False
        
        return n_creations == n_annihilations

    def is_paired(self):
        if len(self) % 2 != 0:
            return False
        for i in range(0, len(self), 2):
            op1 = self.ops[i]
            op2 = self.ops[i + 1]
            if not op1.dagger:
                return False
            if op1.dag() != op2:
                return False
        return True
            
    def num_op_paired(self, sort: bool = True):
        if not self.is_pairable():
            return None, 0
        
        ops_to_pair = list(self.ops)
        pairs = []
        total_sign = 1
        
        while ops_to_pair:
            ann_idx = -1
            for i, op in enumerate(ops_to_pair):
                if not op.dagger:
                    ann_idx = i
                    break
                
            annihilation_op = ops_to_pair[ann_idx]
            creation_op = annihilation_op.dag()
            cre_idx = -1
            
            try:
                # Search for the partner only in the section to the left of the annihilation op
                cre_idx = ops_to_pair.index(creation_op, 0, ann_idx)
            except ValueError:
                # This indicates an unpairable string, e.g., c_i c_i^dagger
                # which should be handled by the initial pairable() check.
                raise ValueError(f"No matching creation operator found for {annihilation_op}.")
            
            num_swaps = ann_idx - cre_idx - 1
            if num_swaps % 2 != 0:
                total_sign *= -1

            pairs.append((ops_to_pair[cre_idx], ops_to_pair[ann_idx]))

            ops_to_pair.pop(ann_idx)
            ops_to_pair.pop(cre_idx)
            
        if sort:
            pairs.sort(key=lambda pair: (pair[0].label, pair[0].subscr))    
        
        paired_string = OpString(tuple(chain.from_iterable(pairs)))
        return paired_string, total_sign
    
    def iter_pairs(self):
        if not self.is_paired():
            raise ValueError("OpString is not paired.")
        
        for i in range(0, len(self), 2):
            yield (self.ops[i].subscr, self.ops[i].label)

class FermionOp:
    def __init__(self, coeffs: list[Expr], ops: list[OpString]):
        self.coeffs = coeffs
        self.ops = ops
        
    def _generate_latex_str(self):
        latex_str = ""
        for coeff, op in zip(self.coeffs, self.ops):
            if coeff.is_Add:
                coeff_latex = f"\\left({sympy.latex(coeff)}\\right)"
            else:
                coeff_latex = sympy.latex(coeff)
            op_latex = op._generate_latex_str().strip()
            if coeff_latex == "1":
                latex_str += f" + {op_latex}"
            elif coeff_latex == "-1":
                latex_str += f" - {op_latex}"
            elif coeff_latex[0] == '-':
                latex_str += f"{coeff_latex} {op_latex}"
            else:
                latex_str += f" + {coeff_latex} {op_latex}"
        if latex_str.startswith(" + "):
            latex_str = latex_str[3:]
        return latex_str
    
    def __repr__(self):
        return f"FermionOp(coeffs={self.coeffs}, ops={self.ops})"

    def _repr_latex_(self):
        latex_str = self._generate_latex_str()
        return f"${latex_str}$"
    
    def __len__(self):
        assert len(self.coeffs) == len(self.ops)
        return len(self.ops)
    
    def __getitem__(self, idx):
        return (self.coeffs[idx], self.ops[idx])
        
    def __add__(self, other):
        if isinstance(other, FermionOp):
            new_coeffs = self.coeffs + other.coeffs
            new_ops = self.ops + other.ops
            return FermionOp(new_coeffs, new_ops)
        return NotImplemented
    
    def __neg__(self):
        new_coeffs = [-coeff for coeff in self.coeffs]
        return FermionOp(new_coeffs, self.ops)
    
    def __sub__(self, other):
        if isinstance(other, FermionOp):
            return self + (-other)
        return NotImplemented
    
    def __rmul__(self, other):
        if isinstance(other, Expr):
            new_coeffs = [other * coeff for coeff in self.coeffs]
            return FermionOp(new_coeffs, self.ops)
        return NotImplemented

    def __mul__(self, other):
        if isinstance(other, Expr):
            new_coeffs = [coeff * other for coeff in self.coeffs]
            return FermionOp(new_coeffs, self.ops)
        if isinstance(other, FermionOp):
            new_coeffs = []
            new_ops = []
            for coeff1, ops1 in zip(self.coeffs, self.ops):
                for coeff2, ops2 in zip(other.coeffs, other.ops):
                    new_coeffs.append(coeff1 * coeff2)
                    new_ops.append(OpString((*ops1.ops, *ops2.ops)))
            return FermionOp(new_coeffs, new_ops)
        return NotImplemented
    
    def dag(self):
        new_coeffs = [sympy.conjugate(coeff) for coeff in self.coeffs]
        new_ops = [op.dag() for op in self.ops]
        return FermionOp(new_coeffs, new_ops)
    
    def eliminate_if(self, condition):
        new_coeffs = []
        new_ops = []
        for coeff, op in zip(self.coeffs, self.ops):
            if not condition(coeff, op):
                new_coeffs.append(coeff)
                new_ops.append(op)
        return FermionOp(new_coeffs, new_ops)
    
    def simplified(self, sympy_simplify: bool = True):
        if not self.ops:
            return FermionOp([], [])
        
        op_map = defaultdict(lambda: sympy.Integer(0))
        for coeff, op in zip(self.coeffs, self.ops):
            op_map[op] += coeff
        
        new_coeffs = []
        new_ops = []
        
        for op, coeff in op_map.items():
            simplified_coeff = sympy.simplify(coeff) if sympy_simplify else coeff
            if simplified_coeff != sympy.Integer(0):
                new_coeffs.append(simplified_coeff)
                new_ops.append(op)
        
        return FermionOp(new_coeffs, new_ops)

    def subs_coeff(self, old: Symbol, new: Expr):
        new_coeffs = []
        for coeff in self.coeffs:
            new_coeff = coeff.subs(old, new)
            new_coeffs.append(new_coeff)
        return FermionOp(new_coeffs, self.ops)

    def subs_op_label(self, old: str, new: str):
        new_ops = []
        for opstr in self.ops:
            new_op_list = []
            for op in opstr.ops:
                if op.label == old:
                    new_op = SingleOp(op.subscr, new, op.dagger)
                else:
                    new_op = deepcopy(op)
                new_op_list.append(new_op)
            new_ops.append(OpString(tuple(new_op_list)))
        return FermionOp(self.coeffs, new_ops)
    
    def is_paired(self):
        return all(op.is_paired() for op in self.ops)
    
    def num_op_paired(self, sort: bool = True):
        new_coeffs = []
        new_ops = []
        
        for coeff, op in zip(self.coeffs, self.ops):
            paired_op, sign = op.num_op_paired(sort=sort)
            
            if paired_op is None:
                raise ValueError(f"OpString {op} is not pairable.")
            
            new_coeffs.append(coeff * sign)
            new_ops.append(paired_op)

        return FermionOp(new_coeffs, new_ops)

In [3]:
def c(subscr: str, label: str) -> FermionOp:
    single_op = SingleOp(subscr, label)
    return FermionOp([sympy.Integer(1)], [OpString([single_op])])

def cdag(subscr: str, label: str) -> FermionOp:
    single_op = SingleOp(subscr, label, dagger=True)
    return FermionOp([sympy.Integer(1)], [OpString([single_op])])

In [4]:
UP = r"\uparrow"
DOWN = r"\downarrow"
UPPER = "+"
LOWER = "-"

qx_str = "q_x"
p_str = "p"
p1_str = "p_1"
p2_str = "p_2"

qx, p, p1, p2 = symbols("q_x p p_1 p_2")
alpha = Function("alpha")
beta = Function("beta")

In [5]:
def expanded_up(label: str) -> FermionOp:
    label_symbol = Symbol(label)
    alpha_conj = sympy.conjugate(alpha(label_symbol))
    beta_conj = sympy.conjugate(beta(label_symbol))
    return alpha_conj * c(UPPER, label) + beta_conj * c(LOWER, label)

def expanded_down(label: str) -> FermionOp:
    label_symbol = Symbol(label)
    alpha_conj = sympy.conjugate(alpha(label_symbol))
    beta_conj = sympy.conjugate(beta(label_symbol))
    return -beta_conj * c(UPPER, label) + alpha_conj * c(LOWER, label)

In [6]:
def elim_cond(_, opstr: OpString):
    return not opstr.is_pairable()

In [7]:
class LatexString:
    def __init__(self, latex_string: str):
        self._latex = latex_string

    def _repr_latex_(self):
        return f"${self._latex}$"
    
    def __str__(self):
        return self._latex
    
def expectation_latex(terms: FermionOp) -> str:
    # convert paired FermionOp terms to LaTeX strings
    if not terms.is_paired():
        raise ValueError("Terms are not paired.")

    latex_str = ""
    for coeff, opstr in terms:
        if coeff.is_Add:
            coeff_latex = f"\\left({sympy.latex(coeff)}\\right)"
        else:
            coeff_latex = sympy.latex(coeff)
        
        num_op_exp_latex = r"\langle "
        for subscr, label in opstr.iter_pairs():
            num_op_exp_latex += f"n_{{{subscr}}}({label}) "
        num_op_exp_latex += r"\rangle"
        
        if coeff_latex == "1":
            latex_str += f" + {num_op_exp_latex}"
        elif coeff_latex == "-1":
            latex_str += f" - {num_op_exp_latex}"
        elif coeff_latex[0] == '-':
            latex_str += f"{coeff_latex} {num_op_exp_latex}"
        else:
            latex_str += f" + {coeff_latex} {num_op_exp_latex}"
    if latex_str.startswith(" + "):
        latex_str = latex_str[3:]
    return LatexString(latex_str)

## Lindbladian

### spin-up

\begin{equation}
    \frac{d}{dt}\langle{n_\uparrow(q_x)}\rangle=-\frac{\gamma}{V\hbar}\sum_{p}\left[\operatorname{Re}\left\langle c_\downarrow^\dagger(p)c_\uparrow^\dagger(q_x) c_\uparrow(q_x)c_\downarrow(p)\right\rangle +\sum_{p}\operatorname{Re}\left\langle c_\downarrow^\dagger(q_x)c_\uparrow^\dagger(p) c_\uparrow(q_x)c_\downarrow(p)\right\rangle\right]
\end{equation}

#### term1
> $$\left\langle c_\downarrow^\dagger(p)c_\uparrow^\dagger(q_x) c_\uparrow(q_x)c_\downarrow(p)\right\rangle$$

- $p\ne q_x$

In [8]:
term1_expanded = expanded_down(p_str).dag() * expanded_up(qx_str).dag() * expanded_up(qx_str) * expanded_down(p_str)
term1_eliminated = term1_expanded.eliminate_if(elim_cond)
expectation_latex(term1_eliminated.num_op_paired().simplified(sympy_simplify=False))

- $p=q_x$

In [9]:
# additional terms for p = qx
term1_p_eq_qx = term1_expanded.subs_coeff(p, qx).subs_op_label(p_str, qx_str).eliminate_if(elim_cond).num_op_paired().simplified()
expectation_latex(term1_p_eq_qx.simplified())

#### term2
> $$\left\langle c_\downarrow^\dagger(q_x)c_\uparrow^\dagger(p) c_\uparrow(q_x)c_\downarrow(p)\right\rangle$$

- $q_x\ne p$

In [10]:
term2_expanded = expanded_down(qx_str).dag() * expanded_up(p_str).dag() * expanded_up(qx_str) * expanded_down(p_str)
term2_eliminated = term2_expanded.eliminate_if(elim_cond)
expectation_latex(term2_eliminated.num_op_paired().simplified(sympy_simplify=False))

- $q_x=p$

In [11]:
# additional terms for p = qx
term2_p_eq_qx = term2_expanded.subs_coeff(p, qx).subs_op_label(p_str, qx_str).eliminate_if(elim_cond).num_op_paired().simplified()
expectation_latex((term2_p_eq_qx).simplified())

### spin-down

\begin{equation}
    \frac{d}{dt}\langle{n_\uparrow(q_x)}\rangle=-\frac{\gamma}{V\hbar}\sum_{p}\left[\operatorname{Re}\left\langle c_\downarrow^\dagger(q_x)c_\uparrow^\dagger(p) c_\uparrow(p)c_\downarrow(q_x)\right\rangle +\sum_{p}\operatorname{Re}\left\langle c_\downarrow^\dagger(p)c_\uparrow^\dagger(q_x) c_\uparrow(p)c_\downarrow(q_x)\right\rangle\right]
\end{equation}

## Non-Hermitian

### spin-up

\begin{align}
	\frac{d}{dt}\langle n_\uparrow(q_x)\rangle =& -\frac{\gamma}{V\hbar}\operatorname{Re}\sum_{p_1 + p_2 = p_3 + p_4} \delta_{q_x, p_2}\langle c_\downarrow^\dagger (p_1) c_\uparrow^\dagger (q_x)c_\uparrow(p_3)c_\downarrow(p_4)\rangle \\
	&-\frac{\gamma}{V\hbar}\operatorname{Re}\sum_{p_1 + p_2 = p_3 + p_4}\langle c_\uparrow^\dagger (q_x) c_\downarrow^\dagger(p_1) c_\uparrow^\dagger (p_2)c_\uparrow(q_x)c_\uparrow(p_3)c_\downarrow(p_4)\rangle \\
	=&-\frac{\gamma}{V\hbar}\operatorname{Re}\sum_p\langle c_\downarrow^\dagger(p)c_\uparrow^\dagger(q_x)c_\uparrow(q_x)c_\downarrow(p)\rangle\\
	&-\frac{\gamma}{V\hbar}\operatorname{Re}\sum_p\langle c_\downarrow^\dagger(p)c_\uparrow^\dagger(q_x)c_\uparrow(p)c_\downarrow(q_x)\rangle\\
	&+\frac{\gamma}{V\hbar}\braket{n_\uparrow(q_x)n_\downarrow(q_x)} \\
	&-\frac{\gamma}{V\hbar}\operatorname{Re}\sum_{p_1, p_2}\braket{c_\uparrow^\dagger(q_x) c_\downarrow^\dagger(p_1) c_\uparrow^\dagger(p_2) c_\uparrow(q_x) c_\uparrow(p_1) c_\downarrow(p_2)}\\
	&-\frac{\gamma}{V\hbar}\operatorname{Re}\sum_{p_1, p_2}\braket{c_\uparrow^\dagger(q_x) c_\downarrow^\dagger(p_2) c_\uparrow^\dagger(p_1) c_\uparrow(q_x) c_\uparrow(p_1) c_\downarrow(p_2)}
\end{align}

#### term1
> $$\braket{c_\uparrow^\dagger(q_x) c_\downarrow^\dagger(p_1) c_\uparrow^\dagger(p_2) c_\uparrow(q_x) c_\uparrow(p_1) c_\downarrow(p_2)}$$

- $q_x, p_1, p_2$ are distinct

In [12]:
term1_expanded = expanded_up(qx_str).dag() * expanded_down(p1_str).dag() * expanded_up(p2_str).dag() * expanded_up(qx_str) * expanded_up(p1_str) * expanded_down(p2_str)
term1_eliminated = term1_expanded.eliminate_if(elim_cond)
expectation_latex(term1_eliminated.num_op_paired().simplified())

- $q_x=p_1\ne p_2$

In [13]:
# additional terms for p1 = qx
term1_p1_eq_qx = term1_expanded.subs_coeff(p1, qx).subs_op_label(p1_str, qx_str).eliminate_if(elim_cond).num_op_paired().simplified()
expectation_latex(term1_p1_eq_qx.simplified())

- $q_x=p_2\ne p_1$

In [14]:
# additional terms for p2 = qx
term1_p2_eq_qx = term1_expanded.subs_coeff(p2, qx).subs_op_label(p2_str, qx_str).eliminate_if(elim_cond).num_op_paired().simplified()
expectation_latex(term1_p2_eq_qx.simplified())

- $p_1=p_2\ne q_x$

In [15]:
# additional terms for p1 = p2
term1_substituted_p1_eq_p2 = term1_expanded.subs_coeff(p1, p).subs_op_label(p1_str, p_str).subs_coeff(p2, p).subs_op_label(p2_str, p_str)
term1_p1_eq_p2 = term1_substituted_p1_eq_p2.subs_coeff(p2, qx).subs_op_label(p2_str, qx_str).eliminate_if(elim_cond).num_op_paired().simplified()
expectation_latex(term1_p1_eq_p2.simplified())

#### term2
> $$\braket{c_\uparrow^\dagger(q_x) c_\downarrow^\dagger(p_2) c_\uparrow^\dagger(p_1) c_\uparrow(q_x) c_\uparrow(p_1) c_\downarrow(p_2)}$$

- $q_x, p_1, p_2$ are distinct

In [16]:
term2_expanded = expanded_up(qx_str).dag() * expanded_down(p2_str).dag() * expanded_up(p1_str).dag() * expanded_up(qx_str) * expanded_up(p1_str) * expanded_down(p2_str)
term2_eliminated = term2_expanded.eliminate_if(elim_cond)
expectation_latex(term2_eliminated.num_op_paired().simplified())

- $q_x=p_1\ne p_2$

In [17]:
# additional terms for p1 = qx
term2_p1_eq_qx = term2_expanded.subs_coeff(p1, qx).subs_op_label(p1_str, qx_str).eliminate_if(elim_cond).num_op_paired()
expectation_latex(term2_p1_eq_qx.simplified())

- $q_x=p_2\ne p_1$

In [18]:
# additional terms for p2 = qx
term2_p2_eq_qx = term2_expanded.subs_coeff(p2, qx).subs_op_label(p2_str, qx_str).eliminate_if(elim_cond).num_op_paired()
expectation_latex(term2_p2_eq_qx.simplified())

- $p_1=p_2\ne q_x$

In [19]:
# additional terms for p1 = p2
term2_substituted_p1_eq_p2 = term2_expanded.subs_coeff(p1, p).subs_op_label(p1_str, p_str).subs_coeff(p2, p).subs_op_label(p2_str, p_str)
term2_p1_eq_p2 = term2_substituted_p1_eq_p2.subs_coeff(p2, qx).subs_op_label(p2_str, qx_str).eliminate_if(elim_cond).num_op_paired()
expectation_latex(term2_p1_eq_p2.simplified())

### spin-down

\begin{align}
	\frac{d}{dt}\langle n_\downarrow(q_x)\rangle =& -\frac{\gamma}{V\hbar}\operatorname{Re}\sum_{p_1 + p_2 = p_3 + p_4} \delta_{q_x, p_1}\langle c_\downarrow^\dagger (q_x) c_\uparrow^\dagger (p_2)c_\uparrow(p_3)c_\downarrow(p_4)\rangle \\
	&-\frac{\gamma}{V\hbar}\operatorname{Re}\sum_{p_1 + p_2 = p_3 + p_4}\langle c_\downarrow^\dagger (q_x) c_\downarrow^\dagger(p_1) c_\uparrow^\dagger (p_2)c_\downarrow(q_x)c_\uparrow(p_3)c_\downarrow(p_4)\rangle\\
	=&-\frac{\gamma}{V\hbar}\operatorname{Re}\sum_p\langle c_\downarrow^\dagger(q_x)c_\uparrow^\dagger(p)c_\uparrow(q_x)c_\downarrow(p)\rangle\\
	&-\frac{\gamma}{V\hbar}\operatorname{Re}\sum_p\langle c_\downarrow^\dagger(q_x)c_\uparrow^\dagger(p)c_\uparrow(p)c_\downarrow(q_x)\rangle\\
	&+\frac{\gamma}{V\hbar}\braket{n_\uparrow(q_x)n_\downarrow(q_x)} \\
	&-\frac{\gamma}{V\hbar}\operatorname{Re}\sum_{p_1, p_2}\braket{c_\downarrow^\dagger(q_x) c_\downarrow^\dagger(p_1) c_\uparrow^\dagger(p_2) c_\downarrow(q_x) c_\uparrow(p_1) c_\downarrow(p_2)}\\
	&-\frac{\gamma}{V\hbar}\operatorname{Re}\sum_{p_1, p_2}\braket{c_\downarrow^\dagger(q_x) c_\downarrow^\dagger(p_2) c_\uparrow^\dagger(p_1) c_\downarrow(q_x) c_\uparrow(p_1) c_\downarrow(p_2)}
\end{align}

#### term1
> $$\braket{c_\downarrow^\dagger(q_x) c_\downarrow^\dagger(p_1) c_\uparrow^\dagger(p_2) c_\downarrow(q_x) c_\uparrow(p_1) c_\downarrow(p_2)}$$

- $q_x, p_1, p_2$ are distinct

In [20]:
term1_expanded = expanded_down(qx_str).dag() * expanded_down(p1_str).dag() * expanded_up(p2_str).dag() * expanded_down(qx_str) * expanded_up(p1_str) * expanded_down(p2_str)
term1_eliminated = term1_expanded.eliminate_if(elim_cond)
expectation_latex(term1_eliminated.num_op_paired().simplified())

- $q_x=p_1\ne p_2$

In [21]:
# additional terms for p1 = qx
term1_p1_eq_qx = term1_expanded.subs_coeff(p1, qx).subs_op_label(p1_str, qx_str).eliminate_if(elim_cond).num_op_paired()
expectation_latex(term1_p1_eq_qx.simplified())

- $q_x=p_2\ne p_1$

In [22]:
# additional terms for p2 = qx
term1_p2_eq_qx = term1_expanded.subs_coeff(p2, qx).subs_op_label(p2_str, qx_str).eliminate_if(elim_cond).num_op_paired()
expectation_latex(term1_p2_eq_qx.simplified())

- $p_1=p_2\ne q_x$

In [23]:
# additional terms for p1 = p2
term1_substituted_p1_eq_p2 = term1_expanded.subs_coeff(p1, p).subs_op_label(p1_str, p_str).subs_coeff(p2, p).subs_op_label(p2_str, p_str)
term1_p1_eq_p2 = term1_substituted_p1_eq_p2.subs_coeff(p2, qx).subs_op_label(p2_str, qx_str).eliminate_if(elim_cond).num_op_paired()
expectation_latex(term1_p1_eq_p2.simplified())

#### term2
> $$\braket{c_\downarrow^\dagger(q_x) c_\downarrow^\dagger(p_2) c_\uparrow^\dagger(p_1) c_\downarrow(q_x) c_\uparrow(p_1) c_\downarrow(p_2)}$$

- $q_x, p_1, p_2$ are distinct

In [24]:
term2_expanded = expanded_down(qx_str).dag() * expanded_down(p2_str).dag() * expanded_up(p1_str).dag() * expanded_down(qx_str) * expanded_up(p1_str) * expanded_down(p2_str)
term2_eliminated = term2_expanded.eliminate_if(elim_cond)
expectation_latex(term2_eliminated.num_op_paired().simplified())

- $q_x=p_1\ne p_2$

In [25]:
# additional terms for p1 = qx
term2_p1_eq_qx = term2_expanded.subs_coeff(p1, qx).subs_op_label(p1_str, qx_str).eliminate_if(elim_cond).num_op_paired()
expectation_latex(term2_p1_eq_qx.simplified())

- $q_x=p_2\ne p_1$

In [26]:
# additional terms for p2 = qx
term2_p2_eq_qx = term2_expanded.subs_coeff(p2, qx).subs_op_label(p2_str, qx_str).eliminate_if(elim_cond).num_op_paired()
expectation_latex(term2_p2_eq_qx.simplified())

- $p_1=p_2\ne q_x$

In [27]:
# additional terms for p1 = p2
term2_substituted_p1_eq_p2 = term2_expanded.subs_coeff(p1, p).subs_op_label(p1_str, p_str).subs_coeff(p2, p).subs_op_label(p2_str, p_str)
term2_p1_eq_p2 = term2_substituted_p1_eq_p2.subs_coeff(p2, qx).subs_op_label(p2_str, qx_str).eliminate_if(elim_cond).num_op_paired()
expectation_latex(term2_p1_eq_p2.simplified())